# Convergent regions and variants

A region converges when it passes Bonferroni and FDR-BH in both the burden test and case-control. Its variants with FDR-BH < 0.05 are saved as the input of the annotation notebooks.

## Setup, imports and configuration

In [ ]:
import numpy as np
import pandas as pd

# Paths
DIR_WSPS = '/home/jupyter/workspace/ws_files'
DIR_NOVA = f'{DIR_WSPS}/Novalis_v3_R12'
DIR_RESU = f'{DIR_NOVA}/Results_union'

# Ancestry / dataset to take forward
ANC, DS = 'EUR', 'WGS'

# Inputs: outputs of the burden test and case-control notebooks
BURDEN_FILE = f'{DIR_RESU}/ALL_BURDEN_FDR.tsv'
CC_FILE     = f'{DIR_RESU}/CaseControl_{ANC}_{DS}_FDRv3.tsv'

# Output: input of the annotation notebooks
VARIANTS_FILE = f'{DIR_RESU}/Convergent_variants_{ANC}_{DS}.tsv'

# Significance thresholds
N_HARS     = 5915              # regions in the union HAR list
BONF       = 0.05 / N_HARS     # 8.45e-6
FDR_THRESH = 0.05

print(f'Selecting convergent regions for {ANC} {DS}')
print(f'  Burden results:       {BURDEN_FILE}')
print(f'  Case-control results: {CC_FILE}')
print(f'  Output:               {VARIANTS_FILE}')

## Load burden and case-control results

In [ ]:
# Burden results for the chosen ancestry and dataset
burden = pd.read_csv(BURDEN_FILE, sep='\t')
burden = burden[(burden['ANCESTRY'] == ANC) & (burden['DATASET'] == DS)].copy()

# Case-control results
cc = pd.read_csv(CC_FILE, sep='\t', low_memory=False)
for col in ('P', 'FDR_BH', 'OR', 'L95', 'U95', 'A1_FREQ'):
    if col in cc.columns:
        cc[col] = pd.to_numeric(cc[col], errors='coerce')

print(f'Burden tests: {len(burden):,} | case-control variants: {len(cc):,}')

## Regions significant in both tests

In [ ]:
burden_hits = burden[(burden['Pvalue'] < BONF) & (burden['FDR_BH'] < FDR_THRESH)]
cc_hits     = cc[(cc['P'] < BONF) & (cc['FDR_BH'] < FDR_THRESH)]

burden_regions = set(burden_hits['GENE'])
cc_regions     = set(cc_hits['HAR'])
convergent     = sorted(burden_regions & cc_regions)

print(f'Burden-significant regions:       {len(burden_regions)}')
print(f'Case-control-significant regions: {len(cc_regions)}')
print(f'Convergent regions:               {len(convergent)}')
if not convergent:
    raise ValueError('No region is significant in both tests; nothing to annotate.')

## Select variants in the convergent regions

In [ ]:
# Variants with FDR-BH < threshold inside the convergent regions
variants = cc[cc['HAR'].isin(convergent) & (cc['FDR_BH'] < FDR_THRESH)].copy()

# Parse REF/ALT from the variant ID if they are not columns
if 'REF' not in variants.columns or variants['REF'].isna().all():
    split_id = variants['ID'].astype(str).str.split(':', expand=True)
    variants['REF'] = split_id.iloc[:, -2].values
    variants['ALT'] = split_id.iloc[:, -1].values

chrom_col = '#CHROM' if '#CHROM' in variants.columns else 'CHROM'
variants['CHROM'] = variants[chrom_col].astype(str).str.replace('chr', '', regex=False)
variants['sig_tier'] = np.where(variants['P'] < BONF, 'Bonf+FDR', 'FDR')

print(f'Variants selected: {len(variants)}')
variants[['HAR', 'CHROM', 'POS', 'REF', 'ALT', 'OR', 'L95', 'U95', 'P', 'FDR_BH', 'sig_tier']]

## Add the best burden result per region

In [ ]:
# Best burden result per region (lowest p across kernels and classes)
burden_best = (burden[burden['GENE'].isin(convergent)]
    .sort_values('Pvalue')
    .groupby('GENE')
    .first()
    .reset_index()
    [['GENE', 'KERNEL', 'CLASS', 'Pvalue', 'FDR_BH']]
    .rename(columns={'GENE': 'HAR', 'KERNEL': 'burden_kernel', 'CLASS': 'burden_class',
                     'Pvalue': 'burden_p', 'FDR_BH': 'burden_fdr'})
)
variants = variants.merge(burden_best, on='HAR', how='left')

burden_best

## Save results

In [ ]:
cols = ['HAR', 'CHROM', 'POS', 'REF', 'ALT', 'sig_tier',
        'OR', 'L95', 'U95', 'P', 'FDR_BH', 'A1_FREQ',
        'burden_kernel', 'burden_class', 'burden_p', 'burden_fdr']
variants[[c for c in cols if c in variants.columns]].to_csv(VARIANTS_FILE, sep='\t', index=False)
print(f'{len(variants)} variants saved: {VARIANTS_FILE}')